In [1]:
!pip install -q transformers accelerate openpyxl

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Carregamento e Preparação dos Dados
df = pd.read_excel("train.xlsx")
df = df.dropna(subset=['resp_text', 'clarity'])
df['resp_text'] = df['resp_text'].astype(str)

# Filtrar as classes e mapeá-las para números
df = df[df['clarity'].isin(['c1', 'c234', 'c5'])].copy()
label_map = {'c1': 0, 'c234': 1, 'c5': 2}
df['label'] = df['clarity'].map(label_map)

# Dividir em Treino e Teste (80% / 20%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['resp_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label'].tolist()
)

# 2. Carregar o Tokenizador do BERTimbau
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenizar os textos 
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

# 3. Criar a Classe de Dataset Customizada
class TransformDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TransformDataset(train_encodings, train_labels)
val_dataset = TransformDataset(val_encodings, val_labels)

# 4. Definir Função de Métricas
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average='macro')
    return {'f1_macro': f1}

# 5. Carregar o Modelo BERT e Configurar o Treinamento
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# 6. Iniciar o Treinamento da Rede Neural
print("Iniciando o treinamento (Fine-Tuning)... Isso pode demorar alguns minutos com GPU.")
trainer.train()

# 7. Avaliação Final
print("\nGerando previsões no conjunto de validação...")
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

# Mostrar o relatório final
target_names = ['c1', 'c234', 'c5']
print("\nRelatório de Classificação BERT:")
print(classification_report(val_labels, preds, target_names=target_names))

print("\nMatriz de Confusão:")
print(confusion_matrix(val_labels, preds))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Iniciando o treinamento (Fine-Tuning)... Isso pode demorar alguns minutos com GPU.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,F1 Macro
1,1.082689,1.076017,0.348847
2,1.035729,1.055044,0.452500
3,0.934169,1.091523,0.446189


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Gerando previsões no conjunto de validação...



Relatório de Classificação BERT:
              precision    recall  f1-score   support

          c1       0.47      0.54      0.50      1269
        c234       0.41      0.31      0.35      1371
          c5       0.48      0.53      0.51      1379

    accuracy                           0.46      4019
   macro avg       0.45      0.46      0.45      4019
weighted avg       0.45      0.46      0.45      4019


Matriz de Confusão:
[[679 283 307]
 [455 424 492]
 [308 335 736]]
